### Chapter 4 - Transforming Data
Reviewing Extract-Transform-Load (ETL) processes
#### Importing and assembling data
**Bad practices:**

- Manually deleting rows at the beginning of the file in spreadsheet software which fails to preserve the edit trail
- The number of columns differs between two files that are going to be appended. Deleting or adding columns fails to preserve the edit trail
- Changing column names when they are not desired
- Overwriting the original file which can have errors with no hope of preserving the original data
- Manual edits are learned and shared orally, not as codified procedure

We are going to load and assemble data from datasets which have some quirks. Goal is to assemble a gas price dataset for two US ports from the Energy Information Administration (EIA). We will load a CSV of gas prices for the US Golf Coast and two worksheets from an Excel workbook for NY Harbor gas prices. The columnms need to be renamed for consistency, then the datasets need to be joined together into a daily time series with three columns: `date`, `gulf_price`, and `ny_price`. Finally, we then save the processed data as a CSV and calculate a correlation coefficient between two gas price variables.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
import os
os.getcwd()

'c:\\Users\\debro\\OneDrive\\Documents\\Public Policy\\Python Exercises'

In [8]:
# Load a CSV with the Gulf Gas Prices
gulf = pd.read_csv("../Data-Science-for-Public-Policy-data-sets/data/doe_usgulf.csv")
# Rename the columns
gulf.columns = ["date", "gulf_price"]
# View the first few rows of the Data
gulf.head(10)

,date,gulf_price
0,1/2/14,2.515
1,1/3/14,2.493
2,1/6/14,2.515
3,1/7/14,2.556
4,1/8/14,2.545
5,1/9/14,2.528
6,1/10/14,2.605
7,1/13/14,2.544
8,1/14/14,2.528
9,1/15/14,2.551


In [17]:
# Load the Excel workbook with NY data
ny1 = pd.read_excel("../Data-Science-for-Public-Policy-data-sets/data/doe_ny.xlsx", sheet_name="part1", skiprows=1).iloc[:, :2]
ny2 = pd.read_excel("../Data-Science-for-Public-Policy-data-sets/data/doe_ny.xlsx", sheet_name="part2", skiprows=1).iloc[:, :2]
# Change the column names
ny1.columns = ny2.columns = ["date", "ny_price"]
# Combine the two DataFrames
ny = pd.concat([ny1, ny2], axis=0, ignore_index=True)
# View the first few rows of the Data
ny.head(10)

,date,ny_price
0,2014-01-02,2.718
1,2014-01-03,2.671
2,2014-01-06,2.678
3,2014-01-07,2.704
4,2014-01-08,2.684
5,2014-01-09,2.667
6,2014-01-10,2.693
7,2014-01-13,2.643
8,2014-01-14,2.636
9,2014-01-15,2.639


In [18]:
gulf.describe()

,gulf_price
count,959.000000
mean,1.744842
std,0.532234
min,0.812000
25%,1.379000
50%,1.562000
75%,1.941000
max,2.953000


In [19]:
ny.describe()

,date,ny_price
count,959,959.000000
mean,2015-11-27 15:03:56.496350464,1.814097
min,2014-01-02 00:00:00,0.947000
25%,2014-12-13 12:00:00,1.449500
50%,2015-11-25 00:00:00,1.604000
75%,2016-11-07 12:00:00,2.042000
max,2017-10-23 00:00:00,3.023000
std,NaN,0.544024


We can bind the two datasets together since they have the same amount of rows. However, we probably want to merge on the columns on `date`. If you were to inspect the datasets, they would have the same data, just in a different datetime format. We can construct a new DataFrame using `pd.concat` and use the `to_csv` function to output the dataset to a CSV.

In [21]:
gas_prices = pd.concat([gulf, ny['ny_price']], axis=1)
gas_prices.to_csv("../data/gas_prices.csv", index=False)
gas_prices.to_json("../data/gas_prices.json", orient="records", lines=True)

#### Manipulating Values
Sometimes we need to clean up data and process the data before analyzing.

In [13]:
# import regular expressions module
import re
import numpy as np
# Create a list of "budget" strings
budget = [
    "Captain's Log, Stardate 1551.8. I have $10.20 for a big galactic map.",
    "The ensign has $1.20 in her pocket.",
    "The ExO has $0.25 left after paying for overpriced warp core fuel.",
    "Chief medical officer is the high roller with $53,13."
]
# Remove the commas for easier processing
bstr = [b.replace(",",".") for b in budget]
print(bstr)
# Extract the dollar amounts using regular expressions
amounts = [float(str(re.findall(r"\$(?:\d{1,3}(?:,\d{3})*|\d+)(?:\.\d{2})?", b)[0])[1:]) for b in bstr]
print(amounts)
# Calcaulate the total amount
print("Total amount available for a galactic big mac is: ${:.2f}".format(np.sum(amounts)))

["Captain's Log. Stardate 1551.8. I have $10.20 for a big galactic map.", 'The ensign has $1.20 in her pocket.', 'The ExO has $0.25 left after paying for overpriced warp core fuel.', 'Chief medical officer is the high roller with $53.13.']
[10.2, 1.2, 0.25, 53.13]
Total amount available for a galactic big mac is: $64.78


#### Text manipulation functions
*Find and replace* funcationality are common in word processing and spreadsheet software, but are not particularly efficient with complex string patterns. Here's a table with text manipulation functions:

| Description | `re`/Base Python | `pandas` |
| :---------- | :----- | :----- |
| Returns either the index position of a matched string or the</br>string containing the matched portion. | `re.search` | `.str.contains()` |
| Returns a logical vector indicating if a matched string was</br>found. | `[<pattern> in i for i <list>]`</br>`[bool(re.search(<pattern>, i)) for i in <list>]` | `str.contains()` |
| Searches for a specified pattern and replaces with user-specified</br>substring. | `re.sub` | `.str.replace('str1', 'str2', regex=True)` |
| Remove matched pattern from string. | `re.sub` | `.str.replace('str1', 'str2', regex=True)` |
| Returns the first position of matched patterns in a string | `re.sub` | `.str.replace('str1', 'str2', regex=True)` |
| Returns the position of all matched patterns in a string | `re.search('pattern', String).start()` | `.str.find('pattern')` |
| Extract substring based on matched pattern. | `re.search(pattern, text).group()` | `.str.extract()` |
| Splits strings into a list of values based on a delimiter. | `re.split(pattern, text)` | `.str.split('pattern')` |
| Extract substring based on start and end positions | `string[start_idx:end_idx]` | `.str.slice(start, stop, step)` |
| Trim whitespace on either end of string (excessive spaces) | `re.sub(r'\s+', ' ', text)`</br>`text.strip()` | `.str.strip()` |
| Returns number of characters in string | `len(string)` | `.str.len()` |
| Returns the number of matched patterns | `len(re.findall(pattern, text))` | `.str.count(pattern)` |
| Convert to upper case | `result1 = re.sub(r'\b\w+\b', lambda match: match.group(0).upper(), text)` | `.str.upper()` |
| Convert to lower case | `result1 = re.sub(r'\b\w+\b', lambda match: match.group(0).lower(), text)` | `.str.lower()` |
| Convert to title case | `re.sub(r"[A-Za-z]+(\'[A-Za-z]+)?", lambda word: word.group(0).capitalize(), string)` | `.str.title()` |
| Pad string (e.g., add leading zeros to string) | `string.ljust(width, fillchar)`</br>`string.rjust(width, fillchar)`</br>`string.center(width, fillchar)` | `.str.pad(side, width, fillchar)` |

#### Regular Expressions (RegEx)
Regex expressions which dictate text patterns are the secret to manipulations.
1. Alternatives (e.g., OR searches) can be surfaced by using a pipe `|`
2. Extent of a search is denoted by parentheses `()`.
3. A search for one specific character should be placed between square brackes `[]`
4. The length of a match is specified using curly brackets `{}`

For example, in NYC, *Broadway* can be written and abbreviated in a number of ways.

In [19]:
# Put together some potential Broadway street name variations and non-Brodway street names
streets = [
    'Bruckner Blvd',
    'Bowery',
    'Broadway',
    'Bway',
    'Bdway',
    'Broad Street',
    'Bridge Street',
    "B'way"
]
# Search for two specific options using regular expressions
options = [s for s in streets if re.search(r'Broadway|Bdway', s)]
# Search for two specific variations using regular expressions
variations = [s for s in streets if re.search(r"B(road|')way", s)]
# Search for cases where either d or apostrophe is between B and way
apostrophe = [s for s in streets if re.search(r"B[d']way", s)]
# Get all Broadway variations even the apostrophe one
all_bway = [s for s in streets if re.search(r"B(road|d|')way", s)]
all_bway

['Broadway', 'Bdway', "B'way"]

**Escaped characters:**
- `\n` new line
- `\r` carriage return
- `\t` tab
- `\'` single quote when a string enclosed in single quotes
- `\"` double quote when a string is enclosed in double quotes
- `\\.` period. Otherwise, un-escaped periods indicates searches for any single character.
- `\\$` dollar sign. A dollar sign without backslashes indicates to find patterns at the end of a string.

**Character classes:**